# Neural Network from Scratch
## Neural networks for energy applications — Day 1, Exercise 2

In this exercise we implement a **2-layer neural network** in pure NumPy to classify sonar signals as Rock or Mine. No deep-learning frameworks are used — we code every component by hand so you understand exactly what happens under the hood.

The dataset (`sonar.all-data.csv`) contains 208 samples with 60 continuous features.

By the end of this notebook you will:
- Implement the sigmoid activation and its derivative
- Initialize weights with proper scaling
- Code forward propagation through two layers
- Compute binary cross-entropy loss
- Implement backpropagation to compute gradients, and check them numerically
- Train the network with gradient descent and evaluate it **once** on the test set
- Explore hyperparameters on a validation split and compare with sklearn's `MLPClassifier`

---
### 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn is used only for splitting, scaling, metrics and the final comparison
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier

np.random.seed(42)
print("Imports complete.")

---
### 2 — Load and Prepare the Sonar Dataset

Same dataset as Lab 1: 60 features, binary label (R/M). We use the same explicit label mapping (**Mine = 1**, the positive class, **Rock = 0**) and the identical 80/20 split call, so the 42 test echoes are the same as in Lab 1 and the scores can be compared on equal terms. Features are standardized with a scaler fitted on the training rows only; we keep the unscaled training rows for the validation experiments in section 12.

In [ ]:
df = pd.read_csv("sonar.all-data.csv", header=None)
feature_cols = [f"feature_{i}" for i in range(60)]
df.columns = feature_cols + ["label"]

X = df[feature_cols].values
y_raw = df["label"].values
LABEL_MAP = {"R": 0, "M": 1}                  # Mine = 1 (positive class), Rock = 0, as in Lab 1
y = df["label"].map(LABEL_MAP).to_numpy()

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Label mapping: {LABEL_MAP}")
print(f"Class distribution: Rock={np.sum(y==0)}, Mine={np.sum(y==1)}")

In [ ]:
# Train/test split: the same call as Lab 1, so these are the same 42 test echoes
idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=0.2, random_state=42, stratify=y_raw
)
X_train_raw, X_test_raw = X[idx_train], X[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

# Standardize features (fit on train, transform both)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# Reshape y to column vectors for the network
y_train = y_train.reshape(-1, 1)
y_test = y_test.reshape(-1, 1)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

---
### 3 — Sigmoid Activation Function

The sigmoid function maps any real number to the range $(0, 1)$:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Its derivative (needed for backpropagation) is:

$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

In [ ]:
def sigmoid(z):
    """Compute the sigmoid activation."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))


def sigmoid_derivative(a):
    """Compute sigmoid derivative given the activation output a = sigmoid(z)."""
    return a * (1.0 - a)


# Quick test
z_test = np.array([-10, -1, 0, 1, 10])
print(f"sigmoid({z_test}) = {sigmoid(z_test).round(4)}")
print(f"sigmoid(0) should be 0.5: {sigmoid(0.0):.4f}")

# Plot
z_range = np.linspace(-6, 6, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z_range, sigmoid(z_range), label="sigmoid(z)", linewidth=2)
ax.plot(z_range, sigmoid_derivative(sigmoid(z_range)), label="sigmoid'(z)",
        linewidth=2, linestyle="--")
ax.set_xlabel("z")
ax.set_ylabel("Output")
ax.set_title("Sigmoid and its Derivative")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
### 4 — Weight Initialization

We initialize weights from a normal distribution scaled by $\frac{1}{\sqrt{n_{\text{in}}}}$ to prevent the activations from exploding or vanishing at the start of training. Biases are initialized to zero.

In [ ]:
def initialize_weights(n_input, n_hidden, n_output):
    """Initialize weights and biases for a 2-layer network.

    Parameters
    ----------
    n_input  : int, number of input features
    n_hidden : int, number of hidden units
    n_output : int, number of output units (1 for binary)

    Returns
    -------
    dict with keys 'W1', 'b1', 'W2', 'b2'
    """
    params = {
        "W1": np.random.randn(n_input, n_hidden) / np.sqrt(n_input),
        "b1": np.zeros((1, n_hidden)),
        "W2": np.random.randn(n_hidden, n_output) / np.sqrt(n_hidden),
        "b2": np.zeros((1, n_output)),
    }
    return params


# Test
test_params = initialize_weights(60, 32, 1)
for k, v in test_params.items():
    print(f"{k}: shape={v.shape}, mean={v.mean():.4f}, std={v.std():.4f}")

---
### 5 — Forward Propagation

Our 2-layer network computes:

**Hidden layer:**
$$z_1 = X \cdot W_1 + b_1, \quad a_1 = \sigma(z_1)$$

**Output layer:**
$$z_2 = a_1 \cdot W_2 + b_2, \quad a_2 = \sigma(z_2)$$

where $a_2$ is the predicted probability.

In [ ]:
def forward(X, params):
    """Perform forward propagation.

    Parameters
    ----------
    X      : np.ndarray, shape (m, n_input)
    params : dict with 'W1', 'b1', 'W2', 'b2'

    Returns
    -------
    cache : dict with 'z1', 'a1', 'z2', 'a2'
    """
    z1 = X @ params["W1"] + params["b1"]
    a1 = sigmoid(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    a2 = sigmoid(z2)

    cache = {"z1": z1, "a1": a1, "z2": z2, "a2": a2}
    return cache


# Test with a small batch
test_cache = forward(X_train[:5], test_params)
print(f"Output shape: {test_cache['a2'].shape}")
print(f"Output values (probabilities): {test_cache['a2'].flatten().round(4)}")

---
### 6 — Binary Cross-Entropy Loss

For binary classification the loss function is:

$$\mathcal{L} = -\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\right]$$

We clip predictions to avoid $\log(0)$.

In [ ]:
def binary_cross_entropy(y_true, y_pred):
    """Compute binary cross-entropy loss.

    Parameters
    ----------
    y_true : np.ndarray, shape (m, 1)
    y_pred : np.ndarray, shape (m, 1) — predicted probabilities

    Returns
    -------
    float
    """
    m = y_true.shape[0]
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss


# Quick test
print(f"Confident, correct (p = 0.99): {binary_cross_entropy(np.array([[1]]), np.array([[0.99]])):.6f}")
print(f"Confident, wrong (p = 0.01):   {binary_cross_entropy(np.array([[1]]), np.array([[0.01]])):.6f}")
print(f"Uninformed (p = 0.5):          {binary_cross_entropy(np.array([[1]]), np.array([[0.50]])):.6f}")

---
### 7 — Backward Propagation

We compute the gradients of the loss with respect to each parameter using the chain rule.

**Output layer gradients:**
$$dz_2 = a_2 - y$$
$$dW_2 = \frac{1}{m} a_1^T \cdot dz_2$$
$$db_2 = \frac{1}{m} \sum dz_2$$

**Hidden layer gradients:**
$$dz_1 = (dz_2 \cdot W_2^T) \odot \sigma'(z_1), \qquad \sigma'(z_1) = a_1 \odot (1 - a_1)$$
$$dW_1 = \frac{1}{m} X^T \cdot dz_1$$
$$db_1 = \frac{1}{m} \sum dz_1$$

`sigmoid_derivative(a1)` computes $\sigma'(z_1)$ from the stored activation $a_1$.

Two checks follow: every gradient must have the shape of its parameter (asserted below), and a **numerical gradient check** (the Part 6 slide) compares the backpropagated gradient with the slope of the loss when one weight is nudged by ±ε. Use float64 arrays (ours are) and restore each perturbed weight. Relative errors around 1e-7 or smaller mean the backward pass is right; errors near 1 mean a bug.

In [ ]:
def backward(X, y, params, cache):
    """Perform backward propagation and return gradients.

    Parameters
    ----------
    X      : np.ndarray, shape (m, n_input)
    y      : np.ndarray, shape (m, 1)
    params : dict with 'W1', 'b1', 'W2', 'b2'
    cache  : dict with 'z1', 'a1', 'z2', 'a2'

    Returns
    -------
    grads : dict with 'dW1', 'db1', 'dW2', 'db2'
    """
    m = X.shape[0]
    a1 = cache["a1"]
    a2 = cache["a2"]

    # Output layer
    dz2 = a2 - y                                      # (m, 1)
    dW2 = (1.0 / m) * (a1.T @ dz2)                   # (n_hidden, 1)
    db2 = (1.0 / m) * np.sum(dz2, axis=0, keepdims=True)  # (1, 1)

    # Hidden layer
    dz1 = (dz2 @ params["W2"].T) * sigmoid_derivative(a1)  # (m, n_hidden)
    dW1 = (1.0 / m) * (X.T @ dz1)                         # (n_input, n_hidden)
    db1 = (1.0 / m) * np.sum(dz1, axis=0, keepdims=True)  # (1, n_hidden)

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads


# Shape check: every gradient must have the shape of its parameter
test_grads = backward(X_train[:5], y_train[:5], test_params,
                      forward(X_train[:5], test_params))
for k, v in test_params.items():
    assert test_grads["d" + k].shape == v.shape, f"d{k}: {test_grads['d' + k].shape} != {v.shape}"
    print(f"d{k}: shape={test_grads['d' + k].shape}  (same as {k})")

In [ ]:
# Numerical gradient check (Part 6 slide): central differences in float64
def grad_check(loss_fn, params, grads, key, idx, eps=1e-5):
    old = params[key][idx]
    params[key][idx] = old + eps
    up = loss_fn(params)
    params[key][idx] = old - eps
    down = loss_fn(params)
    params[key][idx] = old                          # restore the weight
    num, ana = (up - down) / (2 * eps), grads["d" + key][idx]
    return abs(num - ana) / max(abs(num), abs(ana), 1e-12)


np.random.seed(42)
params = initialize_weights(60, 32, 1)              # the network at initialization
assert X_train.dtype == np.float64 and params["W1"].dtype == np.float64
loss_fn = lambda p: binary_cross_entropy(y_train, forward(X_train, p)["a2"])
grads = backward(X_train, y_train, params, forward(X_train, params))
for key, idx in [("W1", (0, 0)), ("b1", (0, 3)), ("W2", (5, 0))]:
    print(key, idx, f"{grad_check(loss_fn, params, grads, key, idx):.1e}")

---
### 8 — Training Loop

We tie everything together: initialize weights, then for each epoch perform forward propagation, compute the loss, run backpropagation, and update the weights.

In [ ]:
def train(X, y, n_hidden, learning_rate, epochs, verbose=True):
    """Train a 2-layer neural network.

    Parameters
    ----------
    X             : np.ndarray, shape (m, n_input)
    y             : np.ndarray, shape (m, 1)
    n_hidden      : int, number of hidden units
    learning_rate : float
    epochs        : int
    verbose       : bool, print progress every 100 epochs

    Returns
    -------
    params       : dict, trained weights
    loss_history : list of float
    """
    n_input = X.shape[1]
    n_output = 1

    # Initialize
    params = initialize_weights(n_input, n_hidden, n_output)
    loss_history = []

    for epoch in range(epochs):
        # Forward
        cache = forward(X, params)

        # Loss
        loss = binary_cross_entropy(y, cache["a2"])
        loss_history.append(loss)

        # Backward
        grads = backward(X, y, params, cache)

        # Update weights
        params["W1"] -= learning_rate * grads["dW1"]
        params["b1"] -= learning_rate * grads["db1"]
        params["W2"] -= learning_rate * grads["dW2"]
        params["b2"] -= learning_rate * grads["db2"]

        if verbose and (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1:5d}/{epochs}  |  Loss: {loss:.6f}")

    return params, loss_history


print("Training function defined.")

---
### 9 — Train the Network

We train with 1000 epochs, a learning rate of 0.01, and 32 hidden units.

In [ ]:
np.random.seed(42)

trained_params, loss_history = train(
    X_train, y_train,
    n_hidden=32,
    learning_rate=0.01,
    epochs=1000,
    verbose=True,
)

---
### 10 — Plot Training Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(loss_history, linewidth=2, color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("Binary Cross-Entropy Loss")
ax.set_title("Training Loss Curve")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Final loss:   {loss_history[-1]:.6f}")

---
### 11 — Evaluate on the Test Set

The settings in section 9 (32 hidden units, learning rate 0.01, 1000 epochs, seed 42) were fixed before looking at any score, so this is a legitimate **one-time** test of a locked choice: the only use of the test set in this notebook. The 42 test echoes are the same as in Lab 1, so compare with the Lab 1 result on equal terms. Each echo is worth 2.4 percentage points.

In [ ]:
def predict(X, params, threshold=0.5):
    """Predict binary labels using the trained network."""
    cache = forward(X, params)
    return (cache["a2"] >= threshold).astype(int)


# Training accuracy
y_pred_train = predict(X_train, trained_params)
train_acc = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {train_acc:.4f} ({int((y_pred_train == y_train).sum())} of {len(y_train)})")

# Test accuracy: the only use of the test set in this notebook
y_pred_test = predict(X_test, trained_params)
test_acc = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy:     {test_acc:.4f} ({int((y_pred_test == y_test).sum())} of {len(y_test)})")

print("\nClassification Report (test set):")
print(classification_report(y_test, y_pred_test, target_names=["Rock", "Mine"]))

---
### 12 — Experiment with Hyperparameters (on a validation split)

Let us see how different choices of hidden layer size, learning rate, and number of epochs affect performance. **These comparisons use a validation split carved from the training rows, not the test set**: choosing settings by test accuracy would turn the test set into a second validation set, and the "best" score would be optimistic.

We hold out 25% of the 166 training rows (42 echoes) as validation data, refit the scaler on the remaining 124 rows, train each setting on those 124 rows, and score it on the validation rows. Validation accuracy on 42 echoes moves in steps of 2.4 points, so we also print the validation loss.

The test score in section 11 belongs to the settings fixed in advance. If a different setting looks better here, report its validation score; do not re-test it on the same 42 test echoes.

In [ ]:
# Validation split carved from the TRAINING rows only; the test rows stay untouched
X_sub_raw, X_val_raw, y_sub, y_val = train_test_split(
    X_train_raw, y_train, test_size=0.25, random_state=42, stratify=y_train.ravel()
)
val_scaler = StandardScaler().fit(X_sub_raw)        # fitted on the 124 sub-training rows only
X_sub = val_scaler.transform(X_sub_raw)
X_val = val_scaler.transform(X_val_raw)
print(f"Sub-training rows: {X_sub.shape[0]}, validation rows: {X_val.shape[0]}")


def run_setting(n_hidden, learning_rate, epochs):
    """Train on the sub-training rows (seed 42); return final training loss, validation loss and accuracy."""
    np.random.seed(42)
    p, losses = train(X_sub, y_sub, n_hidden=n_hidden, learning_rate=learning_rate,
                      epochs=epochs, verbose=False)
    val_loss = binary_cross_entropy(y_val, forward(X_val, p)["a2"])
    val_acc = accuracy_score(y_val, predict(X_val, p))
    return losses[-1], val_loss, val_acc

In [ ]:
# Experiment 1: Vary hidden size
hidden_sizes = [8, 16, 32, 64, 128]
results_hidden = []

print("Varying hidden_size (lr=0.01, epochs=1000), scored on the validation rows")
print("-" * 72)
for h in hidden_sizes:
    train_loss, val_loss, val_acc = run_setting(h, 0.01, 1000)
    results_hidden.append(val_acc)
    print(f"  hidden_size={h:3d}  |  train loss={train_loss:.3f}  val loss={val_loss:.3f}  "
          f"val acc={val_acc:.4f}")

In [ ]:
# Experiment 2: Vary learning rate
learning_rates = [0.001, 0.005, 0.01, 0.05, 0.1]
results_lr = []

print("Varying learning_rate (hidden_size=32, epochs=1000), scored on the validation rows")
print("-" * 72)
for lr in learning_rates:
    train_loss, val_loss, val_acc = run_setting(32, lr, 1000)
    results_lr.append(val_acc)
    print(f"  lr={lr:.4f}  |  train loss={train_loss:.3f}  val loss={val_loss:.3f}  "
          f"val acc={val_acc:.4f}")

In [ ]:
# Experiment 3: Vary epochs
epoch_values = [100, 500, 1000, 2000, 5000]
results_epochs = []

print("Varying epochs (hidden_size=32, lr=0.01), scored on the validation rows")
print("-" * 72)
for ep in epoch_values:
    train_loss, val_loss, val_acc = run_setting(32, 0.01, ep)
    results_epochs.append(val_acc)
    print(f"  epochs={ep:5d}  |  train loss={train_loss:.3f}  val loss={val_loss:.3f}  "
          f"val acc={val_acc:.4f}")

In [ ]:
# Plot all three experiments (validation accuracy)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(hidden_sizes, results_hidden, "o-", color="steelblue", linewidth=2, markersize=8)
axes[0].set_xlabel("Hidden Size")
axes[0].set_ylabel("Validation Accuracy")
axes[0].set_title("Effect of Hidden Size")
axes[0].set_ylim(0.5, 1.05)
axes[0].grid(alpha=0.3)

axes[1].plot(learning_rates, results_lr, "o-", color="coral", linewidth=2, markersize=8)
axes[1].set_xlabel("Learning Rate")
axes[1].set_ylabel("Validation Accuracy")
axes[1].set_title("Effect of Learning Rate")
axes[1].set_xscale("log")
axes[1].set_ylim(0.5, 1.05)
axes[1].grid(alpha=0.3)

axes[2].plot(epoch_values, results_epochs, "o-", color="seagreen", linewidth=2, markersize=8)
axes[2].set_xlabel("Epochs")
axes[2].set_ylabel("Validation Accuracy")
axes[2].set_title("Effect of Epochs")
axes[2].set_ylim(0.5, 1.05)
axes[2].grid(alpha=0.3)

plt.suptitle("Hyperparameter experiments: 124 training rows, 42 validation rows")
plt.tight_layout()
plt.show()

**Reading the experiments.** These are validation scores: they guide a choice, and the best of many noisy scores is an optimistic estimate. Differences of one or two validation echoes (2.4 to 4.8 points) are within noise. Larger learning rates or more epochs keep lowering the training loss, while the validation loss stops improving and then rises again: the overfitting signature. For full-batch gradient descent with small steps, learning rate × epochs largely decides where training ends up (0.001 for 1000 epochs behaves like 0.01 for 100 epochs).

---
### 13 — Compare with sklearn MLPClassifier

As a sanity check, we compare our from-scratch network against sklearn's `MLPClassifier` with a similar architecture. Both are trained on the same 124 sub-training rows and scored on the same validation rows as section 12 (run its first cell before this section). The test set is not used again.

In [ ]:
# Our network, trained on the sub-training rows with the section 9 settings
np.random.seed(42)
our_params, _ = train(X_sub, y_sub, n_hidden=32, learning_rate=0.01, epochs=1000, verbose=False)
our_train_acc = accuracy_score(y_sub, predict(X_sub, our_params))
our_val_acc = accuracy_score(y_val, predict(X_val, our_params))

mlp = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="logistic",    # sigmoid
    solver="sgd",
    learning_rate_init=0.01,
    max_iter=1000,
    random_state=42,
)
mlp.fit(X_sub, y_sub.ravel())
mlp_train_acc = mlp.score(X_sub, y_sub.ravel())
mlp_val_acc = mlp.score(X_val, y_val.ravel())

print(f"{'':20s} {'Train Acc':>12s} {'Val Acc':>12s}")
print("-" * 46)
print(f"{'Our NN (scratch)':20s} {our_train_acc:12.4f} {our_val_acc:12.4f}")
print(f"{'sklearn MLP':20s} {mlp_train_acc:12.4f} {mlp_val_acc:12.4f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
x_pos = np.arange(2)
width = 0.35
ax.bar(x_pos - width/2, [our_train_acc, mlp_train_acc], width,
       label="Train (124 rows)", color="steelblue", edgecolor="k")
ax.bar(x_pos + width/2, [our_val_acc, mlp_val_acc], width,
       label="Validation (42 rows)", color="coral", edgecolor="k")
ax.set_xticks(x_pos)
ax.set_xticklabels(["Our NN (scratch)", "sklearn MLP"])
ax.set_ylabel("Accuracy")
ax.set_title("Scratch NN vs. sklearn MLPClassifier")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

---
### Summary

In this exercise we:

1. Loaded the UCI Sonar dataset with the explicit mapping Mine = 1, Rock = 0, used the same split as Lab 1, and standardized it with a scaler fitted on the training rows.
2. Implemented the **sigmoid** activation and its derivative.
3. Wrote a weight initialization function scaled by $1/\sqrt{n}$.
4. Coded **forward propagation** for a 2-layer network.
5. Implemented **binary cross-entropy** loss.
6. Derived and coded **backpropagation** to compute all four gradients ($dW_1, db_1, dW_2, db_2$), asserted their shapes, and checked them numerically.
7. Built a training loop with gradient descent.
8. Trained the network and plotted the loss curve.
9. Evaluated once on the test set (the same 42 echoes as Lab 1).
10. Explored the effect of hidden size, learning rate, and epochs on a validation split carved from the training rows.
11. Compared our implementation against sklearn's `MLPClassifier` on the same validation rows.

**Key takeaway:** A neural network is fundamentally a sequence of linear transformations and non-linear activations. Backpropagation is just the chain rule applied systematically. Understanding these mechanics prepares you to work with deep-learning frameworks like PyTorch and TensorFlow, where the same operations happen — just with automatic differentiation.